# T1 — Leakage delta: random split vs. cluster-disjoint split

**Question answered.** Does the naive random split (used by most of the literature) inflate
macro-F1 relative to the cluster-disjoint split used in the main paper? This notebook runs
every classifier under both splitting strategies, at the same seeds, and reports the gap
(&Delta;) between them.

**GPU needed.** Yes, for the three CNNs (ResNet-18 / VGG-16 / Inception-v1). The three
classical models (Linear SVM, Random Forest, Logistic Regression) run fine on CPU alone.
Turn on a **GPU T4 x2** accelerator in the notebook settings.

**Estimated runtime (default settings: 6 models x 2 split arms x 5 seeds = 60 fits).**
Classical models: a few minutes total. Each CNN fit: ~2-5 min with early stopping, so
3 CNNs x 2 arms x 5 seeds = 30 fits &asymp; 1.5-2.5 h. Comfortably inside a 12 h session.
If you are short on GPU quota, set `FAST_MODE = True` below (3 seeds, 2 CNNs) or set
`RUN_DEEP_MODELS = False` to get the classical-only leakage delta first (this is the
cheapest, most important part of T1 and finishes in minutes).

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, random, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score, matthews_corrcoef, accuracy_score


### 2. Config

In [ ]:

FAST_MODE        = False          # set True to shrink seeds/models if short on GPU quota
SEEDS             = [0, 1, 2] if FAST_MODE else [0, 1, 2, 3, 4]
RUN_DEEP_MODELS   = True           # set False to get the classical-only delta fast (no GPU needed)
DEEP_MODELS       = ["resnet18", "inception"] if FAST_MODE else ["resnet18", "vgg16", "inception"]
MAX_TFIDF_FEATS   = 512
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)


### 3. Load data (same clean 10-class set as the main experiments)

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH)

le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all  = df["sequence"].values
grp_all = df["cluster"].values
print(f"{len(df)} sequences | {n_cls} classes: {CLASSES}")
print(f"{len(set(grp_all))} MinHash clusters")


### 4. The two split arms

Arm A ignores `cluster` (the naive split most published subtyping papers use). Arm B is the cluster-disjoint split from the main paper -- an entire MinHash cluster always lands on one side.

In [ ]:

def split_arm_A_naive(X, y, seed):
    """Arm A: stratified k-fold, sequences shuffled freely -- groups/clusters ignored.
    This is what a random train_test_split or plain StratifiedKFold does, and it is the
    split strategy used (usually implicitly) by most of the alignment-free literature."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf.split(X, y))
    # carve a validation set out of the training fold, also naive
    skf2 = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
    tr2, va2 = next(skf2.split(X[tr_idx], y[tr_idx]))
    return tr_idx[tr2], tr_idx[va2], te_idx

def split_arm_B_cluster_disjoint(X, y, grp, seed):
    """Arm B: StratifiedGroupKFold -- no MinHash cluster straddles train/test."""
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te


### 5. Encoders and classical models

In [ ]:

def fit_tfidf(X_train_seqs, max_features=MAX_TFIDF_FEATS):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(6, 6), max_features=max_features,
                           lowercase=False)
    Xtr = vec.fit_transform(X_train_seqs)
    return vec, Xtr

def encode_split(seq_tr, seq_va, seq_te):
    vec, Xtr = fit_tfidf(seq_tr)
    Xva = vec.transform(seq_va); Xte = vec.transform(seq_te)
    return Xtr, Xva, Xte

def eval_preds(y_true, y_pred):
    return dict(
        f1      = f1_score(y_true, y_pred, average="macro"),
        bal_acc = balanced_accuracy_score(y_true, y_pred),
        mcc     = matthews_corrcoef(y_true, y_pred),
        acc     = accuracy_score(y_true, y_pred),
    )

def run_classical(model_name, Xtr, ytr, Xte, yte, seed):
    cw = "balanced"
    if model_name == "svm":
        clf = LinearSVC(class_weight=cw, random_state=seed, max_iter=20000)
    elif model_name == "logreg":
        clf = LogisticRegression(class_weight=cw, random_state=seed, max_iter=5000)
    elif model_name == "rf":
        clf = RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=seed, n_jobs=-1)
    else:
        raise ValueError(model_name)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    return eval_preds(yte, pred)


### 6. Deep models (1D CNNs over TF-IDF 6-mer feature vectors)

Re-implemented here to match the architecture family used throughout the paper (Section 2.5): a plain VGG-style stack, a ResNet-style stack with residual connections and stochastic depth, and an Inception-style stack with parallel branches. If your original per-model notebooks differ in exact layer widths, that is fine for this ablation -- T1 only needs the **same** model compared across the two split arms, not an exact replica of the main-paper architecture.

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Model, Input

def se_block(x, filters):
    return x  # placeholder hook, unused by default -- kept for easy extension

def conv_bn_relu(x, filters, kernel=5, stride=1):
    x = layers.Conv1D(filters, kernel, strides=stride, padding="same")(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)

def build_vgg16_1d(input_len, n_classes):
    inp = Input(shape=(input_len, 1))
    x = inp
    for filters, n_convs in [(32,2),(64,2),(128,3),(256,3),(256,3)]:
        for _ in range(n_convs):
            x = conv_bn_relu(x, filters, 3)
        x = layers.MaxPooling1D(2, padding="same")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return Model(inp, out, name="vgg16_1d")

class StochasticDepth(layers.Layer):
    def __init__(self, p=0.85, **kw):
        super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training:
            return x
        r = tf.random.uniform([tf.shape(x)[0], 1, 1])
        return x * tf.floor(self.p + r) / self.p
    def get_config(self):
        return {**super().get_config(), "p": self.p}

def res_block(x, filters, stride=1):
    shortcut = x
    y = conv_bn_relu(x, filters, 3, stride)
    y = layers.Conv1D(filters, 3, padding="same")(y)
    y = layers.BatchNormalization()(y)
    if shortcut.shape[-1] != filters or stride != 1:
        shortcut = layers.Conv1D(filters, 1, strides=stride, padding="same")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    y = StochasticDepth(0.85)(y)
    out = layers.Add()([shortcut, y])
    return layers.Activation("relu")(out)

def build_resnet18_1d(input_len, n_classes):
    inp = Input(shape=(input_len, 1))
    x = conv_bn_relu(inp, 32, 7, 2)
    x = layers.MaxPooling1D(3, strides=2, padding="same")(x)
    for filters, stride in [(32,1),(32,1),(64,2),(64,1),(128,2),(128,1),(256,2),(256,1)]:
        x = res_block(x, filters, stride)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return Model(inp, out, name="resnet18_1d")

def inception_module(x, f1x1, f3x3, f5x5, fpool):
    b1 = conv_bn_relu(x, f1x1, 1)
    b2 = conv_bn_relu(x, f3x3, 3)
    b3 = conv_bn_relu(x, f5x5, 5)
    b4 = layers.MaxPooling1D(3, strides=1, padding="same")(x)
    b4 = conv_bn_relu(b4, fpool, 1)
    return layers.Concatenate()([b1, b2, b3, b4])

def build_inception_1d(input_len, n_classes):
    inp = Input(shape=(input_len, 1))
    x = conv_bn_relu(inp, 32, 7, 2)
    x = layers.MaxPooling1D(3, strides=2, padding="same")(x)
    x = inception_module(x, 16, 24, 8, 8)
    x = inception_module(x, 32, 40, 16, 16)
    x = layers.MaxPooling1D(2, padding="same")(x)
    x = inception_module(x, 40, 48, 20, 20)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return Model(inp, out, name="inception_v1_1d")

BUILDERS = {"vgg16": build_vgg16_1d, "resnet18": build_resnet18_1d, "inception": build_inception_1d}

def run_deep(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, epochs=60, batch=48):
    tf.random.set_seed(seed)
    Xtr_d = np.asarray(Xtr.todense() if hasattr(Xtr, "todense") else Xtr, dtype=np.float32)[..., None]
    Xva_d = np.asarray(Xva.todense() if hasattr(Xva, "todense") else Xva, dtype=np.float32)[..., None]
    Xte_d = np.asarray(Xte.todense() if hasattr(Xte, "todense") else Xte, dtype=np.float32)[..., None]
    mu, sd = Xtr_d.mean(), Xtr_d.std() + 1e-8
    Xtr_d = (Xtr_d - mu) / sd; Xva_d = (Xva_d - mu) / sd; Xte_d = (Xte_d - mu) / sd

    model = BUILDERS[model_name](Xtr_d.shape[1], n_cls)
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                   metrics=["accuracy"])
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    model.fit(Xtr_d, ytr, validation_data=(Xva_d, yva), epochs=epochs, batch_size=batch,
              class_weight=cw, callbacks=[es], verbose=0)
    pred = np.argmax(model.predict(Xte_d, verbose=0), axis=1)
    tf.keras.backend.clear_session()
    return eval_preds(yte, pred)


### 7. Main loop — every model, both arms, every seed

In [ ]:

rows = []
CLASSICAL = ["svm", "rf", "logreg"]

for seed in SEEDS:
    random.seed(seed); np.random.seed(seed)
    for arm_name, splitter in [("random", split_arm_A_naive), ("cluster_disjoint", split_arm_B_cluster_disjoint)]:
        if arm_name == "random":
            tr, va, te = splitter(X_all, y_all, seed)
        else:
            tr, va, te = splitter(X_all, y_all, grp_all, seed)
        ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
        Xtr_tf, Xva_tf, Xte_tf = encode_split(X_all[tr], X_all[va], X_all[te])

        for m in CLASSICAL:
            metrics = run_classical(m, Xtr_tf, ytr, Xte_tf, yte, seed)
            rows.append({"model": m, "arm": arm_name, "seed": seed, **metrics})
            print(seed, arm_name, m, metrics)

        if RUN_DEEP_MODELS:
            for m in DEEP_MODELS:
                metrics = run_deep(m, Xtr_tf, ytr, Xva_tf, yva, Xte_tf, yte, seed)
                rows.append({"model": m, "arm": arm_name, "seed": seed, **metrics})
                print(seed, arm_name, m, metrics)

        pd.DataFrame(rows).to_csv(f"{OUT}/t1_leakage_delta_raw.csv", index=False)  # incremental save


### 8. Aggregate: Table with random vs. cluster-disjoint macro-F1 and Delta

In [ ]:

raw = pd.DataFrame(rows)
summ = raw.groupby(["model", "arm"])[["f1", "bal_acc", "mcc", "acc"]].agg(["mean", "std"]).round(4)
summ.columns = [f"{a}_{b}" for a, b in summ.columns]
summ = summ.reset_index()

pivot_f1 = summ.pivot(index="model", columns="arm", values="f1_mean")
pivot_f1["delta_f1_random_minus_cd"] = pivot_f1["random"] - pivot_f1["cluster_disjoint"]
print(pivot_f1.round(4))

summ.to_csv(f"{OUT}/t1_leakage_delta_summary.csv", index=False)
pivot_f1.to_csv(f"{OUT}/t1_leakage_delta_table2_columns.csv")
print("\nInterpretation: positive delta_f1 means the naive random split over-reports macro-F1")
print("relative to the honest cluster-disjoint split, i.e. leakage is present and inflating scores.")
print("If delta is near zero for every model, go to the T2 notebook next -- the MinHash")
print("clustering threshold is probably too strict and is not catching real near-duplicates.")


### 9. Plot

In [ ]:

fig, ax = plt.subplots(figsize=(7,4.5))
models = pivot_f1.index.tolist()
x = np.arange(len(models))
ax.bar(x-0.18, pivot_f1["random"], width=0.36, label="Random split (Arm A)")
ax.bar(x+0.18, pivot_f1["cluster_disjoint"], width=0.36, label="Cluster-disjoint split (Arm B)")
ax.set_xticks(x); ax.set_xticklabels(models, rotation=20)
ax.set_ylabel("macro-F1"); ax.set_ylim(0,1.02)
ax.legend(); ax.set_title("T1: leakage delta, random vs. cluster-disjoint split")
plt.tight_layout(); plt.savefig(f"{OUT}/t1_leakage_delta.png", dpi=200)
plt.show()


### 10. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r t1_leakage_delta_outputs.zip t1_leakage_delta_raw.csv t1_leakage_delta_summary.csv t1_leakage_delta_table2_columns.csv t1_leakage_delta.png